In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import ndcg_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import json
import logging
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
import asyncio
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import pickle
import os
from scipy import stats
import nest_asyncio

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@dataclass
class RerankingExample:
    """Training example for reranking model"""
    query: str
    document_text: str
    doc_id: str
    relevance_score: float  # 0-1 relevance score
    metadata_features: Dict[str, Any] = field(default_factory=dict)
    
@dataclass
class RerankingBatch:
    """Batch of documents for a single query"""
    query: str
    documents: List[RerankingExample]
    query_id: str = ""

@dataclass
class TrainingMetrics:
    """Metrics tracked during training"""
    epoch: int
    train_loss: float
    val_loss: float
    ndcg_at_5: float
    ndcg_at_10: float
    map_score: float
    precision_at_5: float
    recall_at_5: float

class LegalRerankingDataset(Dataset):
    """PyTorch dataset for legal document reranking"""
    
    def __init__(self, 
                 batches: List[RerankingBatch], 
                 tokenizer, 
                 max_length: int = 512,
                 include_metadata: bool = True):
        self.batches = batches
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.include_metadata = include_metadata
        
        # Flatten batches into individual examples
        self.examples = []
        for batch in batches:
            for doc in batch.documents:
                self.examples.append((batch.query, doc))
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        query, document = self.examples[idx]
        
        # Tokenize query-document pair
        query_doc_text = f"[CLS] {query} [SEP] {document.document_text}"
        
        encoding = self.tokenizer(
            query_doc_text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Extract metadata features if available
        metadata_features = torch.zeros(10)  # Fixed size feature vector
        if self.include_metadata and document.metadata_features:
            # Court level (0-1 normalized)
            court_level = document.metadata_features.get('court_level', 0)
            if court_level:
                metadata_features[0] = 1.0 / court_level  # Supreme=1.0, High=0.5, District=0.33
            
            # Citation count (normalized)
            citation_count = document.metadata_features.get('citation_count', 0)
            metadata_features[1] = min(citation_count / 50.0, 1.0)
            
            # Authority score
            metadata_features[2] = document.metadata_features.get('authority_score', 0.0)
            
            # Recency boost
            metadata_features[3] = document.metadata_features.get('recency_boost', 0.0)
            
            # Document type (0 for case, 1 for act)
            metadata_features[4] = 1.0 if document.metadata_features.get('doc_type') == 'act' else 0.0
            
            # Legal domain match count
            metadata_features[5] = document.metadata_features.get('domain_match_count', 0.0) / 5.0
            
            # Jurisdiction match
            metadata_features[6] = 1.0 if document.metadata_features.get('jurisdiction_match', False) else 0.0
            
            # Act references count
            act_refs = document.metadata_features.get('act_references', 0)
            metadata_features[7] = min(act_refs / 10.0, 1.0)
            
            # Judge count
            judge_count = document.metadata_features.get('judge_count', 0)
            metadata_features[8] = min(judge_count / 5.0, 1.0)
            
            # Vector similarity score
            metadata_features[9] = document.metadata_features.get('vector_score', 0.0)
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'metadata_features': metadata_features,
            'relevance_score': torch.tensor(document.relevance_score, dtype=torch.float),
            'doc_id': document.doc_id,
            'query': query
        }

class NeuralLegalReranker(nn.Module):
    """Neural reranking model for legal documents"""
    
    def __init__(self, 
                 model_name: str = "nlpaueb/legal-bert-base-uncased",
                 metadata_dim: int = 10,
                 hidden_dim: int = 256,
                 dropout_rate: float = 0.1):
        super().__init__()
        
        try:
            # Try legal BERT first, fallback to regular BERT if not available
            self.bert = AutoModel.from_pretrained(model_name)
        except:
            logger.warning(f"Could not load {model_name}, falling back to bert-base-uncased")
            self.bert = AutoModel.from_pretrained("bert-base-uncased")
            
        self.bert_dim = self.bert.config.hidden_size
        
        # Metadata processing
        self.metadata_processor = nn.Sequential(
            nn.Linear(metadata_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim // 2, hidden_dim // 4)
        )
        
        # Combined processing
        combined_dim = self.bert_dim + hidden_dim // 4
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()  # Output relevance score 0-1
        )
        
        # Layer normalization
        self.layer_norm = nn.LayerNorm(combined_dim)
        
    def forward(self, input_ids, attention_mask, metadata_features):
        # BERT encoding
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        bert_pooled = bert_outputs.pooler_output  # [CLS] token representation
        
        # Process metadata features
        metadata_processed = self.metadata_processor(metadata_features)
        
        # Combine BERT and metadata features
        combined_features = torch.cat([bert_pooled, metadata_processed], dim=-1)
        combined_features = self.layer_norm(combined_features)
        
        # Final classification
        relevance_score = self.classifier(combined_features)
        
        return relevance_score.squeeze()

class RerankingTrainer:
    """Trainer for the neural reranking model"""
    
    def __init__(self, 
                 model: NeuralLegalReranker,
                 train_loader: DataLoader,
                 val_loader: DataLoader,
                 learning_rate: float = 2e-5,
                 weight_decay: float = 1e-5,
                 device: str = None):
        
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = model.to(self.device)
        
        self.train_loader = train_loader
        self.val_loader = val_loader
        
        # Optimizer and loss
        self.optimizer = optim.AdamW(
            self.model.parameters(), 
            lr=learning_rate, 
            weight_decay=weight_decay
        )
        
        # Use MSE loss for regression, but we'll also implement ranking loss
        self.mse_loss = nn.MSELoss()
        
        # Training history
        self.training_history = {
            'train_loss': [],
            'val_loss': [],
            'ndcg_at_5': [],
            'ndcg_at_10': [],
            'map_score': [],
            'precision_at_5': [],
            'recall_at_5': []
        }
        
        # Learning rate scheduler
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', patience=3, factor=0.5, verbose=True
        )
    
    def pairwise_ranking_loss(self, predictions, relevance_scores, margin=1.0):
        """Pairwise ranking loss for better ranking performance"""
        batch_size = predictions.size(0)
        loss = 0.0
        num_pairs = 0
        
        for i in range(batch_size):
            for j in range(i + 1, batch_size):
                # If doc i is more relevant than doc j
                if relevance_scores[i] > relevance_scores[j]:
                    # We want pred_i > pred_j
                    diff = predictions[j] - predictions[i] + margin
                    loss += torch.clamp(diff, min=0.0)
                    num_pairs += 1
                elif relevance_scores[j] > relevance_scores[i]:
                    # We want pred_j > pred_i
                    diff = predictions[i] - predictions[j] + margin
                    loss += torch.clamp(diff, min=0.0)
                    num_pairs += 1
        
        return loss / max(num_pairs, 1)
    
    def train_epoch(self):
        """Train for one epoch"""
        self.model.train()
        total_loss = 0.0
        num_batches = 0
        
        for batch in self.train_loader:
            # Move to device
            input_ids = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            metadata_features = batch['metadata_features'].to(self.device)
            relevance_scores = batch['relevance_score'].to(self.device)
            
            # Forward pass
            predictions = self.model(input_ids, attention_mask, metadata_features)
            
            # Combined loss: MSE + Pairwise ranking
            mse = self.mse_loss(predictions, relevance_scores)
            ranking_loss = self.pairwise_ranking_loss(predictions, relevance_scores)
            loss = mse + 0.5 * ranking_loss
            
            # Backward pass
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        return total_loss / num_batches
    
    def evaluate(self, loader: DataLoader):
        """Evaluate the model"""
        self.model.eval()
        total_loss = 0.0
        all_predictions = []
        all_relevance = []
        all_queries = []
        all_doc_ids = []
        
        with torch.no_grad():
            for batch in loader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                metadata_features = batch['metadata_features'].to(self.device)
                relevance_scores = batch['relevance_score'].to(self.device)
                
                predictions = self.model(input_ids, attention_mask, metadata_features)
                
                mse = self.mse_loss(predictions, relevance_scores)
                ranking_loss = self.pairwise_ranking_loss(predictions, relevance_scores)
                loss = mse + 0.5 * ranking_loss
                
                total_loss += loss.item()
                
                # Store predictions for metric calculation
                all_predictions.extend(predictions.cpu().numpy())
                all_relevance.extend(relevance_scores.cpu().numpy())
                all_queries.extend(batch['query'])
                all_doc_ids.extend(batch['doc_id'])
        
        avg_loss = total_loss / len(loader)
        
        # Calculate ranking metrics
        metrics = self.calculate_ranking_metrics(
            all_predictions, all_relevance, all_queries, all_doc_ids
        )
        
        return avg_loss, metrics
    
    def calculate_ranking_metrics(self, predictions, relevance_scores, queries, doc_ids):
        """Calculate ranking evaluation metrics"""
        # Group by query
        query_groups = defaultdict(list)
        for pred, rel, query, doc_id in zip(predictions, relevance_scores, queries, doc_ids):
            query_groups[query].append({
                'prediction': pred,
                'relevance': rel,
                'doc_id': doc_id
            })
        
        ndcg_5_scores = []
        ndcg_10_scores = []
        map_scores = []
        precision_5_scores = []
        recall_5_scores = []
        
        for query, docs in query_groups.items():
            if len(docs) < 2:
                continue
                
            # Sort by predicted relevance (descending)
            docs_sorted = sorted(docs, key=lambda x: x['prediction'], reverse=True)
            
            # Extract relevance scores in sorted order
            true_relevance = [doc['relevance'] for doc in docs_sorted]
            predicted_order = list(range(len(docs_sorted)))
            
            # NDCG@5 and NDCG@10
            if len(true_relevance) >= 5:
                ndcg_5 = ndcg_score([true_relevance], [predicted_order], k=5)
                ndcg_5_scores.append(ndcg_5)
            
            if len(true_relevance) >= 10:
                ndcg_10 = ndcg_score([true_relevance], [predicted_order], k=10)
                ndcg_10_scores.append(ndcg_10)
            
            # MAP (Mean Average Precision)
            ap_score = self.average_precision(true_relevance)
            map_scores.append(ap_score)
            
            # Precision@5 and Recall@5
            relevant_docs = sum(1 for rel in true_relevance if rel > 0.5)  # Binary relevance threshold
            if relevant_docs > 0:
                top_5_relevant = sum(1 for rel in true_relevance[:5] if rel > 0.5)
                precision_5 = top_5_relevant / min(5, len(true_relevance))
                recall_5 = top_5_relevant / relevant_docs
                
                precision_5_scores.append(precision_5)
                recall_5_scores.append(recall_5)
        
        return {
            'ndcg_at_5': np.mean(ndcg_5_scores) if ndcg_5_scores else 0.0,
            'ndcg_at_10': np.mean(ndcg_10_scores) if ndcg_10_scores else 0.0,
            'map_score': np.mean(map_scores) if map_scores else 0.0,
            'precision_at_5': np.mean(precision_5_scores) if precision_5_scores else 0.0,
            'recall_at_5': np.mean(recall_5_scores) if recall_5_scores else 0.0
        }
    
    def average_precision(self, relevance_scores, threshold=0.5):
        """Calculate Average Precision for a single query"""
        relevant_count = 0
        precision_sum = 0.0
        
        for i, score in enumerate(relevance_scores):
            if score > threshold:
                relevant_count += 1
                precision_at_i = relevant_count / (i + 1)
                precision_sum += precision_at_i
        
        total_relevant = sum(1 for score in relevance_scores if score > threshold)
        return precision_sum / total_relevant if total_relevant > 0 else 0.0
    
    def train(self, num_epochs: int, save_path: str = None, early_stopping_patience: int = 5):
        """Train the model with early stopping"""
        best_val_loss = float('inf')
        patience_counter = 0
        
        logger.info(f"Starting training for {num_epochs} epochs on {self.device}")
        
        for epoch in range(num_epochs):
            # Training
            train_loss = self.train_epoch()
            
            # Validation
            val_loss, val_metrics = self.evaluate(self.val_loader)
            
            # Update learning rate
            self.scheduler.step(val_loss)
            
            # Store metrics
            self.training_history['train_loss'].append(train_loss)
            self.training_history['val_loss'].append(val_loss)
            self.training_history['ndcg_at_5'].append(val_metrics['ndcg_at_5'])
            self.training_history['ndcg_at_10'].append(val_metrics['ndcg_at_10'])
            self.training_history['map_score'].append(val_metrics['map_score'])
            self.training_history['precision_at_5'].append(val_metrics['precision_at_5'])
            self.training_history['recall_at_5'].append(val_metrics['recall_at_5'])
            
            # Logging
            logger.info(f"Epoch {epoch+1}/{num_epochs}:")
            logger.info(f"  Train Loss: {train_loss:.4f}")
            logger.info(f"  Val Loss: {val_loss:.4f}")
            logger.info(f"  NDCG@5: {val_metrics['ndcg_at_5']:.4f}")
            logger.info(f"  NDCG@10: {val_metrics['ndcg_at_10']:.4f}")
            logger.info(f"  MAP: {val_metrics['map_score']:.4f}")
            logger.info(f"  P@5: {val_metrics['precision_at_5']:.4f}")
            logger.info(f"  R@5: {val_metrics['recall_at_5']:.4f}")
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                
                # Save best model
                if save_path:
                    self.save_model(save_path)
                    logger.info(f"Saved best model to {save_path}")
            else:
                patience_counter += 1
                
            if patience_counter >= early_stopping_patience:
                logger.info(f"Early stopping at epoch {epoch+1}")
                break
        
        return self.training_history
    
    def save_model(self, path: str):
        """Save model and training history"""
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'training_history': self.training_history
        }, path)
    
    def load_model(self, path: str):
        """Load model and training history"""
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.training_history = checkpoint.get('training_history', self.training_history)
    
    def plot_training_curves(self, save_path: str = None):
        """Plot training curves"""
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        # Loss curves
        axes[0, 0].plot(self.training_history['train_loss'], label='Train Loss')
        axes[0, 0].plot(self.training_history['val_loss'], label='Val Loss')
        axes[0, 0].set_title('Loss Curves')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].legend()
        
        # NDCG curves
        axes[0, 1].plot(self.training_history['ndcg_at_5'], label='NDCG@5')
        axes[0, 1].plot(self.training_history['ndcg_at_10'], label='NDCG@10')
        axes[0, 1].set_title('NDCG Curves')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('NDCG')
        axes[0, 1].legend()
        
        # MAP curve
        axes[0, 2].plot(self.training_history['map_score'])
        axes[0, 2].set_title('Mean Average Precision')
        axes[0, 2].set_xlabel('Epoch')
        axes[0, 2].set_ylabel('MAP')
        
        # Precision curve
        axes[1, 0].plot(self.training_history['precision_at_5'])
        axes[1, 0].set_title('Precision@5')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Precision@5')
        
        # Recall curve
        axes[1, 1].plot(self.training_history['recall_at_5'])
        axes[1, 1].set_title('Recall@5')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Recall@5')
        
        # Combined metrics
        axes[1, 2].plot(self.training_history['ndcg_at_5'], label='NDCG@5')
        axes[1, 2].plot(self.training_history['map_score'], label='MAP')
        axes[1, 2].plot(self.training_history['precision_at_5'], label='P@5')
        axes[1, 2].set_title('Combined Metrics')
        axes[1, 2].set_xlabel('Epoch')
        axes[1, 2].set_ylabel('Score')
        axes[1, 2].legend()
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

class NeuralRerankingSystem:
    """Complete neural reranking system for legal documents"""
    
    def __init__(self, model_path: str = None, device: str = None):
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        
        try:
            self.tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
        except:
            logger.warning("Could not load legal BERT tokenizer, using BERT base")
            self.tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
        
        # Initialize model
        self.model = NeuralLegalReranker()
        
        if model_path and os.path.exists(model_path):
            self.load_model(model_path)
            logger.info(f"Loaded model from {model_path}")
        else:
            logger.info("Initialized new model")
        
        self.model.to(self.device)
        self.model.eval()
    
    def load_model(self, path: str):
        """Load trained model"""
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
    
    async def rerank_documents(self, 
                             query: str, 
                             documents: List[Dict[str, Any]], 
                             top_k: int = 20) -> List[Dict[str, Any]]:
        """Rerank documents using the trained neural model"""
        
        if not documents:
            return []
        
        # Prepare input data
        reranking_examples = []
        for doc in documents:
            example = RerankingExample(
                query=query,
                document_text=doc.get('content', ''),
                doc_id=doc.get('doc_id', ''),
                relevance_score=0.0,  # Will be predicted
                metadata_features=self._extract_metadata_features(doc)
            )
            reranking_examples.append(example)
        
        # Create dataset and dataloader
        batch = RerankingBatch(query=query, documents=reranking_examples)
        dataset = LegalRerankingDataset([batch], self.tokenizer, include_metadata=True)
        dataloader = DataLoader(dataset, batch_size=len(documents), shuffle=False)
        
        # Get predictions
        self.model.eval()
        predictions = []
        
        with torch.no_grad():
            for batch_data in dataloader:
                input_ids = batch_data['input_ids'].to(self.device)
                attention_mask = batch_data['attention_mask'].to(self.device)
                metadata_features = batch_data['metadata_features'].to(self.device)
                
                batch_predictions = self.model(input_ids, attention_mask, metadata_features)
                predictions.extend(batch_predictions.cpu().numpy())
        
        # Add predictions to documents and sort
        for i, doc in enumerate(documents):
            doc['neural_score'] = float(predictions[i])
            doc['original_index'] = i
        
        # Sort by neural score
        reranked_docs = sorted(documents, key=lambda x: x['neural_score'], reverse=True)
        
        return reranked_docs[:top_k]
    
    def _extract_metadata_features(self, doc: Dict[str, Any]) -> Dict[str, Any]:
        """Extract metadata features from document"""
        metadata = doc.get('metadata', {})
        
        return {
            'court_level': metadata.get('court_level', 0),
            'citation_count': metadata.get('citation_count', 0),
            'authority_score': metadata.get('authority_score', 0.0),
            'recency_boost': metadata.get('recency_boost', 0.0),
            'doc_type': metadata.get('doc_type', 'case'),
            'domain_match_count': len(set(metadata.get('legal_domains', [])) & 
                                    set(doc.get('query_domains', []))),
            'jurisdiction_match': metadata.get('jurisdiction', '').lower() in 
                                doc.get('query', '').lower(),
            'act_references': metadata.get('act_references', 0),
            'judge_count': metadata.get('judge_count', 0),
            'vector_score': doc.get('vector_score', 0.0)
        }

class TrainingDataGenerator:
    """Generate training data for the reranking model"""
    
    def __init__(self):
        self.legal_queries = [
            "contempt of court proceedings against judicial officers",
            "constitutional provisions for fundamental rights protection",
            "criminal procedure for arrest and detention",
            "contract law breach of agreement remedies",
            "tort liability for negligence damages",
            "property rights and title disputes",
            "administrative law judicial review procedures",
            "evidence law admissibility of digital documents",
            "family law custody and maintenance provisions",
            "commercial law company registration requirements"
        ]
        
        self.legal_domains = ['constitutional', 'criminal', 'civil', 'commercial', 'administrative']
        
    def generate_synthetic_training_data(self, num_queries: int = 100, docs_per_query: int = 20) -> List[RerankingBatch]:
        """Generate synthetic training data"""
        batches = []
        
        for i in range(num_queries):
            query = np.random.choice(self.legal_queries)
            query_id = f"query_{i}"
            
            documents = []
            for j in range(docs_per_query):
                # Generate relevance score (biased towards lower scores for realism)
                relevance = np.random.beta(2, 5)  # Beta distribution skewed towards 0
                
                # Generate document text (simplified)
                doc_text = f"Legal document {j} discussing {query.split()[0]} with relevance {relevance:.2f}"
                
                # Generate metadata features
                metadata_features = {
                    'court_level': np.random.choice([1, 2, 3], p=[0.1, 0.3, 0.6]),
                    'citation_count': np.random.poisson(5),
                    'authority_score': np.random.uniform(0, 1),
                    'recency_boost': np.random.uniform(0, 1),
                    'doc_type': np.random.choice(['case', 'act']),
                    'domain_match_count': np.random.randint(0, 3),
                    'jurisdiction_match': np.random.choice([True, False]),
                    'act_references': np.random.poisson(2),
                    'judge_count': np.random.randint(1, 4),
                    'vector_score': np.random.uniform(0.3, 0.9)
                }
                
                example = RerankingExample(
                    query=query,
                    document_text=doc_text,
                    doc_id=f"doc_{i}_{j}",
                    relevance_score=relevance,
                    metadata_features=metadata_features
                )
                documents.append(example)
            
            batch = RerankingBatch(query=query, documents=documents, query_id=query_id)
            batches.append(batch)
        
        return batches
    
    def save_training_data(self, batches: List[RerankingBatch], path: str):
        """Save training data to file"""
        data = []
        for batch in batches:
            batch_data = {
                'query': batch.query,
                'query_id': batch.query_id,
                'documents': []
            }
            for doc in batch.documents:
                doc_data = {
                    'query': doc.query,
                    'document_text': doc.document_text,
                    'doc_id': doc.doc_id,
                    'relevance_score': doc.relevance_score,
                    'metadata_features': doc.metadata_features
                }
                batch_data['documents'].append(doc_data)
            data.append(batch_data)
        
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
    
    def load_training_data(self, path: str) -> List[RerankingBatch]:
        """Load training data from file"""
        with open(path, 'r') as f:
            data = json.load(f)
        
        batches = []
        for batch_data in data:
            documents = []
            for doc_data in batch_data['documents']:
                example = RerankingExample(
                    query=doc_data['query'],
                    document_text=doc_data['document_text'],
                    doc_id=doc_data['doc_id'],
                    relevance_score=doc_data['relevance_score'],
                    metadata_features=doc_data['metadata_features']
                )
                documents.append(example)
            
            batch = RerankingBatch(
                query=batch_data['query'],
                documents=documents,
                query_id=batch_data['query_id']
            )
            batches.append(batch)
        
        return batches

class RerankingEvaluator:
    """Comprehensive evaluator for reranking system performance"""
    
    def __init__(self):
        self.evaluation_results = []
        
    def evaluate_system(self, 
                       neural_system: NeuralRerankingSystem,
                       test_batches: List[RerankingBatch],
                       baseline_method: str = "vector_score") -> Dict[str, Any]:
        """Evaluate neural reranking system against baseline"""
        
        results = {
            'neural_metrics': defaultdict(list),
            'baseline_metrics': defaultdict(list),
            'improvement': {},
            'per_query_results': []
        }
        
        logger.info(f"Evaluating on {len(test_batches)} test queries...")
        
        for batch in test_batches:
            # Prepare documents for neural reranking
            documents = []
            for doc in batch.documents:
                doc_dict = {
                    'doc_id': doc.doc_id,
                    'content': doc.document_text,
                    'metadata': doc.metadata_features,
                    'vector_score': doc.metadata_features.get('vector_score', 0.5),
                    'true_relevance': doc.relevance_score
                }
                documents.append(doc_dict)
            
            # Neural reranking
            neural_ranked = asyncio.run(neural_system.rerank_documents(
                batch.query, documents.copy(), top_k=len(documents)
            ))
            
            # Baseline reranking (by vector score)
            baseline_ranked = sorted(documents.copy(), 
                                   key=lambda x: x[baseline_method], reverse=True)
            
            # Calculate metrics for both
            neural_metrics = self._calculate_query_metrics(neural_ranked, 'neural_score')
            baseline_metrics = self._calculate_query_metrics(baseline_ranked, baseline_method)
            
            # Store results
            for metric, value in neural_metrics.items():
                results['neural_metrics'][metric].append(value)
            
            for metric, value in baseline_metrics.items():
                results['baseline_metrics'][metric].append(value)
            
            # Per-query results
            query_result = {
                'query': batch.query,
                'neural_ndcg_5': neural_metrics['ndcg_at_5'],
                'baseline_ndcg_5': baseline_metrics['ndcg_at_5'],
                'improvement': neural_metrics['ndcg_at_5'] - baseline_metrics['ndcg_at_5'],
                'neural_map': neural_metrics['map'],
                'baseline_map': baseline_metrics['map']
            }
            results['per_query_results'].append(query_result)
        
        # Calculate average improvements
        for metric in results['neural_metrics'].keys():
            neural_avg = np.mean(results['neural_metrics'][metric])
            baseline_avg = np.mean(results['baseline_metrics'][metric])
            improvement = ((neural_avg - baseline_avg) / baseline_avg) * 100 if baseline_avg > 0 else 0
            
            results['improvement'][metric] = {
                'neural_avg': neural_avg,
                'baseline_avg': baseline_avg,
                'improvement_pct': improvement
            }
        
        return results
    
    def _calculate_query_metrics(self, ranked_docs: List[Dict], score_key: str) -> Dict[str, float]:
        """Calculate ranking metrics for a single query"""
        if not ranked_docs:
            return {k: 0.0 for k in ['ndcg_at_5', 'ndcg_at_10', 'map', 'precision_at_5', 'recall_at_5']}
        
        # Extract true relevance scores in ranked order
        true_relevance = [doc['true_relevance'] for doc in ranked_docs]
        predicted_scores = [doc.get(score_key, 0.0) for doc in ranked_docs]
        
        # NDCG scores
        ndcg_5 = ndcg_score([true_relevance], [predicted_scores], k=5) if len(true_relevance) >= 5 else 0.0
        ndcg_10 = ndcg_score([true_relevance], [predicted_scores], k=10) if len(true_relevance) >= 10 else 0.0
        
        # MAP
        map_score = self._calculate_ap(true_relevance)
        
        # Precision and Recall at 5
        relevant_threshold = 0.5
        total_relevant = sum(1 for rel in true_relevance if rel > relevant_threshold)
        top_5_relevant = sum(1 for rel in true_relevance[:5] if rel > relevant_threshold)
        
        precision_5 = top_5_relevant / min(5, len(true_relevance)) if len(true_relevance) > 0 else 0.0
        recall_5 = top_5_relevant / total_relevant if total_relevant > 0 else 0.0
        
        return {
            'ndcg_at_5': ndcg_5,
            'ndcg_at_10': ndcg_10,
            'map': map_score,
            'precision_at_5': precision_5,
            'recall_at_5': recall_5
        }
    
    def _calculate_ap(self, relevance_scores: List[float], threshold: float = 0.5) -> float:
        """Calculate Average Precision"""
        relevant_count = 0
        precision_sum = 0.0
        
        for i, score in enumerate(relevance_scores):
            if score > threshold:
                relevant_count += 1
                precision_at_i = relevant_count / (i + 1)
                precision_sum += precision_at_i
        
        total_relevant = sum(1 for score in relevance_scores if score > threshold)
        return precision_sum / total_relevant if total_relevant > 0 else 0.0
    
    def print_evaluation_report(self, results: Dict[str, Any]):
        """Print comprehensive evaluation report"""
        print(f"\n{'='*80}")
        print("NEURAL RERANKING EVALUATION REPORT")
        print('='*80)
        
        print("\nOVERALL PERFORMANCE COMPARISON:")
        print("-" * 50)
        
        for metric, data in results['improvement'].items():
            print(f"{metric.upper()}:")
            print(f"  Neural:     {data['neural_avg']:.4f}")
            print(f"  Baseline:   {data['baseline_avg']:.4f}")
            print(f"  Improvement: {data['improvement_pct']:+.2f}%")
            print()
        
        print("\nTOP 5 MOST IMPROVED QUERIES:")
        print("-" * 50)
        sorted_queries = sorted(results['per_query_results'], 
                               key=lambda x: x['improvement'], reverse=True)
        
        for i, query_result in enumerate(sorted_queries[:5]):
            print(f"{i+1}. Query: {query_result['query'][:60]}...")
            print(f"   NDCG@5 improvement: {query_result['improvement']:+.4f}")
            print(f"   Neural: {query_result['neural_ndcg_5']:.4f} | "
                  f"Baseline: {query_result['baseline_ndcg_5']:.4f}")
            print()
        
        print("\nBOTTOM 5 QUERIES (NEED IMPROVEMENT):")
        print("-" * 50)
        
        for i, query_result in enumerate(sorted_queries[-5:]):
            print(f"{i+1}. Query: {query_result['query'][:60]}...")
            print(f"   NDCG@5 improvement: {query_result['improvement']:+.4f}")
            print(f"   Neural: {query_result['neural_ndcg_5']:.4f} | "
                  f"Baseline: {query_result['baseline_ndcg_5']:.4f}")
            print()
        
        # Statistical significance test
        neural_ndcg = [r['neural_ndcg_5'] for r in results['per_query_results']]
        baseline_ndcg = [r['baseline_ndcg_5'] for r in results['per_query_results']]
        
        t_stat, p_value = stats.ttest_rel(neural_ndcg, baseline_ndcg)
        
        print(f"\nSTATISTICAL SIGNIFICANCE TEST:")
        print("-" * 50)
        print(f"Paired t-test p-value: {p_value:.6f}")
        print(f"Statistical significance: {'Yes' if p_value < 0.05 else 'No'} (α=0.05)")
        print(f"Effect size (Cohen's d): {self._cohens_d(neural_ndcg, baseline_ndcg):.4f}")
    
    def _cohens_d(self, x, y):
        """Calculate Cohen's d for effect size"""
        nx = len(x)
        ny = len(y)
        dof = nx + ny - 2
        return (np.mean(x) - np.mean(y)) / np.sqrt(((nx-1)*np.std(x, ddof=1)**2 + (ny-1)*np.std(y, ddof=1)**2) / dof)
    
    def plot_comparison_charts(self, results: Dict[str, Any], save_path: str = None):
        """Create visualization charts for evaluation results"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. Metric comparison bar chart
        metrics = list(results['improvement'].keys())
        neural_scores = [results['improvement'][m]['neural_avg'] for m in metrics]
        baseline_scores = [results['improvement'][m]['baseline_avg'] for m in metrics]
        
        x = np.arange(len(metrics))
        width = 0.35
        
        axes[0, 0].bar(x - width/2, neural_scores, width, label='Neural', alpha=0.8)
        axes[0, 0].bar(x + width/2, baseline_scores, width, label='Baseline', alpha=0.8)
        axes[0, 0].set_xlabel('Metrics')
        axes[0, 0].set_ylabel('Score')
        axes[0, 0].set_title('Performance Comparison by Metric')
        axes[0, 0].set_xticks(x)
        axes[0, 0].set_xticklabels([m.replace('_', '\n') for m in metrics], rotation=45)
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Improvement percentage chart
        improvements = [results['improvement'][m]['improvement_pct'] for m in metrics]
        colors = ['green' if imp > 0 else 'red' for imp in improvements]
        
        axes[0, 1].bar(metrics, improvements, color=colors, alpha=0.7)
        axes[0, 1].set_xlabel('Metrics')
        axes[0, 1].set_ylabel('Improvement (%)')
        axes[0, 1].set_title('Percentage Improvement over Baseline')
        axes[0, 1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
        axes[0, 1].tick_params(axis='x', rotation=45)
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. Per-query NDCG@5 scatter plot
        neural_ndcg = [r['neural_ndcg_5'] for r in results['per_query_results']]
        baseline_ndcg = [r['baseline_ndcg_5'] for r in results['per_query_results']]
        
        axes[1, 0].scatter(baseline_ndcg, neural_ndcg, alpha=0.6)
        axes[1, 0].plot([0, 1], [0, 1], 'r--', alpha=0.8, label='Equal performance')
        axes[1, 0].set_xlabel('Baseline NDCG@5')
        axes[1, 0].set_ylabel('Neural NDCG@5')
        axes[1, 0].set_title('Per-Query NDCG@5 Comparison')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Improvement distribution histogram
        improvements_per_query = [r['improvement'] for r in results['per_query_results']]
        
        axes[1, 1].hist(improvements_per_query, bins=20, alpha=0.7, edgecolor='black')
        axes[1, 1].axvline(x=0, color='red', linestyle='--', alpha=0.8, label='No improvement')
        axes[1, 1].axvline(x=np.mean(improvements_per_query), color='green', 
                          linestyle='-', alpha=0.8, label=f'Mean: {np.mean(improvements_per_query):.4f}')
        axes[1, 1].set_xlabel('NDCG@5 Improvement')
        axes[1, 1].set_ylabel('Number of Queries')
        axes[1, 1].set_title('Distribution of Per-Query Improvements')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

# Main execution function
async def main():
    """Main function to demonstrate training and evaluation"""
    
    print("Starting Neural Legal Reranker Training and Evaluation")
    print("=" * 60)
    
    # 1. Generate training data
    print("\n1. Generating training data...")
    data_generator = TrainingDataGenerator()
    training_batches = data_generator.generate_synthetic_training_data(
        num_queries=150, docs_per_query=25
    )
    
    # Save training data
    data_generator.save_training_data(training_batches, 'training_data.json')
    print(f"Generated {len(training_batches)} training batches")
    
    # 2. Split data
    train_batches, val_test_batches = train_test_split(training_batches, test_size=0.4, random_state=42)
    val_batches, test_batches = train_test_split(val_test_batches, test_size=0.5, random_state=42)
    
    print(f"Split: Train={len(train_batches)}, Val={len(val_batches)}, Test={len(test_batches)}")
    
    # 3. Create datasets and dataloaders
    print("\n2. Creating datasets...")
    try:
        tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
    except:
        logger.warning("Could not load legal BERT tokenizer, using BERT base")
        tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    
    train_dataset = LegalRerankingDataset(train_batches, tokenizer)
    val_dataset = LegalRerankingDataset(val_batches, tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    print(f"Train dataset: {len(train_dataset)} examples")
    print(f"Val dataset: {len(val_dataset)} examples")
    
    # 4. Initialize model and trainer
    print("\n3. Initializing model...")
    model = NeuralLegalReranker()
    trainer = RerankingTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        learning_rate=2e-5
    )
    
    # 5. Train model
    print("\n4. Starting training...")
    training_history = trainer.train(
        num_epochs=10,
        save_path='best_reranker_model.pth',
        early_stopping_patience=3
    )
    
    # 6. Plot training curves
    print("\n5. Plotting training curves...")
    trainer.plot_training_curves('training_curves.png')
    
    # 7. Evaluate on test set
    print("\n6. Evaluating on test set...")
    neural_system = NeuralRerankingSystem(model_path='best_reranker_model.pth')
    evaluator = RerankingEvaluator()
    
    evaluation_results = evaluator.evaluate_system(
        neural_system=neural_system,
        test_batches=test_batches,
        baseline_method="vector_score"
    )
    
    # 8. Print evaluation report
    print("\n7. Evaluation Results:")
    evaluator.print_evaluation_report(evaluation_results)
    
    # 9. Create comparison charts
    print("\n8. Creating comparison charts...")
    evaluator.plot_comparison_charts(evaluation_results, 'evaluation_comparison.png')
    
    print("\nNeural reranking system setup complete!")
    
    return {
        'model': neural_system,
        'trainer': trainer,
        'evaluator': evaluator,
        'results': evaluation_results
    }

# Demo function for standalone usage
def demo_usage():
    """Demonstrate how to use the neural reranking system"""
    
    print("=" * 60)
    print("NEURAL RERANKING SYSTEM DEMO")
    print("=" * 60)
    
    # Initialize system
    neural_system = NeuralRerankingSystem()
    
    # Mock documents for demonstration
    sample_documents = [
        {
            'doc_id': 'doc_1',
            'content': 'Supreme Court ruling on contempt of court procedures and judicial authority',
            'metadata': {
                'court_level': 1,
                'citation_count': 15,
                'authority_score': 0.9,
                'recency_boost': 0.3,
                'doc_type': 'case',
                'legal_domains': ['constitutional', 'procedural'],
                'jurisdiction': 'Supreme Court',
                'act_references': 2,
                'judge_count': 3
            },
            'vector_score': 0.85
        },
        {
            'doc_id': 'doc_2',
            'content': 'High Court decision on administrative law and judicial review procedures',
            'metadata': {
                'court_level': 2,
                'citation_count': 8,
                'authority_score': 0.7,
                'recency_boost': 0.6,
                'doc_type': 'case',
                'legal_domains': ['administrative'],
                'jurisdiction': 'High Court',
                'act_references': 1,
                'judge_count': 2
            },
            'vector_score': 0.72
        },
        {
            'doc_id': 'doc_3',
            'content': 'Constitutional provisions regarding fundamental rights and judicial protection',
            'metadata': {
                'court_level': 1,
                'citation_count': 25,
                'authority_score': 0.95,
                'recency_boost': 0.1,
                'doc_type': 'act',
                'legal_domains': ['constitutional'],
                'jurisdiction': 'Supreme Court',
                'act_references': 0,
                'judge_count': 0
            },
            'vector_score': 0.78
        }
    ]
    
    # Demo query
    query = "contempt of court proceedings against judicial officers"
    
    print(f"\nQuery: {query}")
    print(f"Original documents: {len(sample_documents)}")
    
    # Rerank documents
    async def run_demo():
        reranked = await neural_system.rerank_documents(query, sample_documents, top_k=3)
        
        print(f"\nReranked results:")
        for i, doc in enumerate(reranked):
            print(f"{i+1}. Doc ID: {doc['doc_id']}")
            print(f"   Neural Score: {doc['neural_score']:.4f}")
            print(f"   Vector Score: {doc['vector_score']:.4f}")
            print(f"   Content: {doc['content'][:80]}...")
            print()
        
        return reranked
    
    # Run demo
    return asyncio.run(run_demo())

if __name__ == "__main__":
    # For Jupyter notebook compatibility
    nest_asyncio.apply()
    
    # You can run either the full training pipeline or just the demo
    print("Choose option:")
    print("1. Run full training pipeline")
    print("2. Run demo with pre-initialized system")
    
    # For demo purposes, we'll run the demo
    print("\nRunning demo...")
    demo_results = demo_usage()
    
    # To run full training, uncomment the line below:
    # full_results = asyncio.run(main())

Choose option:
1. Run full training pipeline
2. Run demo with pre-initialized system

Running demo...
NEURAL RERANKING SYSTEM DEMO


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

INFO:__main__:Initialized new model



Query: contempt of court proceedings against judicial officers
Original documents: 3

Reranked results:
1. Doc ID: doc_1
   Neural Score: 0.4963
   Vector Score: 0.8500
   Content: Supreme Court ruling on contempt of court procedures and judicial authority...

2. Doc ID: doc_3
   Neural Score: 0.4860
   Vector Score: 0.7800
   Content: Constitutional provisions regarding fundamental rights and judicial protection...

3. Doc ID: doc_2
   Neural Score: 0.4847
   Vector Score: 0.7200
   Content: High Court decision on administrative law and judicial review procedures...



model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]